# 11 — Deepfake Detection

Generation artifacts from GAN and diffusion models leave detectable signatures.

## GAN and Diffusion Artifacts

Deepfake detection exploits generation-specific artifacts:

**GAN artifacts**:
- Spectral artifacts: GANs leave periodic patterns in the frequency domain (GAN fingerprints, Frank et al. 2020)
- Checkerboard artifacts from transposed convolutions with even kernel/stride combinations
- Blending seams at face boundaries in face-swap methods

**Diffusion artifacts**:
- Diffusion models produce more photorealistic outputs with fewer checkerboard artifacts
- But they introduce subtle high-frequency patterns from the iterative denoising process
- JPEG compression interacts with diffusion artifacts in detectable ways
- Semantic inconsistencies: incorrect eye reflections, mismatched lighting, impossible hand geometry

**Detection approaches**:
1. *CNN-based*: trained classifier on raw pixels (high accuracy but not robust to post-processing)
2. *Frequency analysis*: FFT/DCT spectrum analysis — GAN fingerprints survive JPEG
3. *GAN fingerprinting*: each GAN architecture leaves a characteristic spectral signature
4. *LLM-era detectors*: multi-modal models trained on watermarked vs unwatermarked content (C2PA metadata)

In [ ]:
# Deepfake detector: frequency + CNN features
import numpy as np
import torch
import torch.nn as nn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

np.random.seed(42)

# Simulate real vs GAN-generated image features
def simulate_real_image(n=28):
    """Natural image: dominated by low frequencies, power law spectrum."""
    img = np.zeros((n, n))
    for freq in range(1, 10):
        img += (1/freq) * np.random.randn() * np.sin(
            2*np.pi*freq * np.outer(np.linspace(0,1,n), np.linspace(0,1,n))
        )
    return img + np.random.randn(n, n) * 0.05

def simulate_gan_image(n=28):
    """GAN image: natural + periodic checkerboard artifact."""
    img = simulate_real_image(n)
    # Add periodic artifact at Nyquist/2
    artifact_freq = n // 4
    artifact = 0.1 * np.sin(2*np.pi*artifact_freq * np.outer(
        np.arange(n)/n, np.ones(n)
    ))
    return img + artifact

def simulate_diffusion_image(n=28):
    """Diffusion image: less periodic, subtle high-freq patterns."""
    img = simulate_real_image(n)
    # High-frequency noise from denoising
    hf_noise = np.random.randn(n, n) * 0.03
    # Apply slight spatial correlation
    from scipy.ndimage import gaussian_filter
    try:
        from scipy.ndimage import gaussian_filter as gf
        hf_noise = hf_noise - gf(hf_noise, sigma=1)
    except ImportError:
        pass
    return img + hf_noise

def extract_frequency_features(img):
    """Extract frequency domain features for deepfake detection."""
    fft = np.fft.fft2(img)
    mag = np.abs(np.fft.fftshift(fft))
    log_mag = np.log1p(mag)

    n = img.shape[0]
    center = n // 2

    # Radial frequency bins
    features = []
    for r in range(1, n//2, n//8):
        ring_mask = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                d = np.sqrt((i-center)**2 + (j-center)**2)
                if r <= d < r + n//8:
                    ring_mask[i, j] = 1
        if ring_mask.sum() > 0:
            features.append(log_mag[ring_mask.astype(bool)].mean())
            features.append(log_mag[ring_mask.astype(bool)].std())

    # Peak frequency
    features.append(log_mag.max())
    features.append(log_mag.mean())
    return np.array(features)

# Generate dataset
n_each = 300
X_list, y_list = [], []
for _ in range(n_each):
    X_list.append(extract_frequency_features(simulate_real_image()))
    y_list.append(0)  # real
for _ in range(n_each):
    X_list.append(extract_frequency_features(simulate_gan_image()))
    y_list.append(1)  # fake

X = np.array(X_list)
y = np.array(y_list)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
detector = GradientBoostingClassifier(n_estimators=50, random_state=42)
detector.fit(X_train, y_train)

print('Deepfake detector results:')
print(classification_report(y_test, detector.predict(X_test),
                             target_names=['real', 'fake']))

In [ ]:
# Visualise frequency signatures
import matplotlib.pyplot as plt

real_img = simulate_real_image()
gan_img = simulate_gan_image()

def get_fft_spectrum(img):
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img))))

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].imshow(real_img, cmap='gray')
axes[0, 0].set_title('Real Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(gan_img, cmap='gray')
axes[0, 1].set_title('GAN Image')
axes[0, 1].axis('off')

axes[1, 0].imshow(get_fft_spectrum(real_img), cmap='hot')
axes[1, 0].set_title('Real FFT Spectrum')
axes[1, 0].axis('off')

axes[1, 1].imshow(get_fft_spectrum(gan_img), cmap='hot')
axes[1, 1].set_title('GAN FFT Spectrum (periodic artifacts)')
axes[1, 1].axis('off')

plt.suptitle('Deepfake Detection: Frequency Signatures')
plt.tight_layout()
plt.savefig('/tmp/deepfake_freq.png', dpi=100, bbox_inches='tight')
plt.show()
print('Frequency signature comparison saved')